In [ ]:
from __future__ import annotations

import argparse
import json
import random
import re
import sys
import time
from pathlib import Path
from typing import Iterable, List, Optional

import numpy as np
import pandas as pd
import requests

# ================================================================================
# 1. CONFIG
# ================================================================================

RANDOM_SEED = 42

SOURCES = [
    {"name": "IEDB", "base_url": "https://query-api.iedb.org"},
    {"name": "CEDAR", "base_url": "https://cedar-api.iedb.org"},
]

# --- PRIMARY targets: specific antigen, UniProt-verified -----------------------
PRIMARY_TARGETS = [
    {"protein_name": "SARS-CoV-2 Spike", "organism": "SARS-CoV-2 (Severe acute respiratory syndrome coronavirus 2)",
     "uniprot_id": "P0DTC2", "gene_symbol": "S"},
    {"protein_name": "Influenza A Nucleoprotein (NP)", "organism": "Influenza A virus (A/Puerto Rico/8/1934 H1N1)",
     "uniprot_id": "P03466", "gene_symbol": "NP"},
    {"protein_name": "Influenza A Hemagglutinin (HA)", "organism": "Influenza A virus (A/Puerto Rico/8/1934 H1N1)",
     "uniprot_id": "P03452", "gene_symbol": "HA"},
    {"protein_name": "HIV-1 Gag", "organism": "Human immunodeficiency virus 1 (HIV-1, isolate HXB2)",
     "uniprot_id": "P04591", "gene_symbol": "gag"},
    {"protein_name": "M. tuberculosis Ag85A", "organism": "Mycobacterium tuberculosis (strain ATCC 25618 / H37Rv)",
     "uniprot_id": "P9WQP3", "gene_symbol": "fbpA"},
    {"protein_name": "M. tuberculosis ESAT-6", "organism": "Mycobacterium tuberculosis (strain ATCC 25618 / H37Rv)",
     "uniprot_id": "P9WNK7", "gene_symbol": "esxA"},
    {"protein_name": "M. tuberculosis CFP-10", "organism": "Mycobacterium tuberculosis (strain ATCC 25618 / H37Rv)",
     "uniprot_id": "P9WNK5", "gene_symbol": "esxB"},
    {"protein_name": "EBV EBNA1", "organism": "Epstein-Barr virus (Human herpesvirus 4)",
     "uniprot_id": "P03211", "gene_symbol": "EBNA1"},
    {"protein_name": "EBV LMP1", "organism": "Epstein-Barr virus (Human herpesvirus 4)",
     "uniprot_id": "P03230", "gene_symbol": "LMP1"},
    {"protein_name": "EBV LMP2", "organism": "Epstein-Barr virus (Human herpesvirus 4)",
     "uniprot_id": "P13285", "gene_symbol": "LMP2"},
    {"protein_name": "EBV BZLF1", "organism": "Epstein-Barr virus (Human herpesvirus 4)",
     "uniprot_id": "P03206", "gene_symbol": "BZLF1"},
    {"protein_name": "EBV GP350/GP340", "organism": "Epstein-Barr virus (Human herpesvirus 4)",
     "uniprot_id": "P03200", "gene_symbol": "BLLF1"},
]

# --- EXTERNAL targets: organism-level (no single UniProt antigen) --------------
EXTERNAL_ORGANISMS = [
    {"organism": "Dengue virus", "organism_query": "Dengue virus"},
    {"organism": "Zika virus", "organism_query": "Zika virus"},
    {"organism": "Hepatitis C virus", "organism_query": "Hepatitis C virus"},
    {"organism": "Human respiratory syncytial virus", "organism_query": "respiratory syncytial virus"},
    {"organism": "Coronavirus HKU15", "organism_query": "Coronavirus HKU15"},
]

MIN_LENGTH = 8
MAX_LENGTH = 20
SLEEP_SECONDS = 0.25
PAGE_SIZE = 10000
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

ENDPOINT_ASSAY_LABELS = {
    "epitope_search": "Epitope (unspecified assay)",
    "tcell_search": "T cell assay",
    "bcell_search": "B cell assay",
    "mhc_search": "MHC ligand assay",
}

# --- MHC reference panel (matches the 13 alleles found in the target CSV) ------
MHC_I_ALLELES = ["HLA-A*01:01", "HLA-A*02:01", "HLA-A*03:01", "HLA-A*24:02", "HLA-B*07:02", "HLA-B*44:03"]
MHC_I_LENGTHS = [9, 10]
MHC_II_ALLELES = ["HLA-DRB1*01:01", "HLA-DRB1*03:01", "HLA-DRB1*04:01", "HLA-DRB1*07:01",
                   "HLA-DRB1*11:01", "HLA-DRB1*13:02", "HLA-DRB1*15:01"]

IEDB_MHCI_TOOL_URL = "http://tools-cluster-interface.iedb.org/tools_api/mhci/"
IEDB_MHCII_TOOL_URL = "http://tools-cluster-interface.iedb.org/tools_api/mhcii/"

TARGET_TOTAL_PER_CLASS = 1000     # final balanced dataset: this many Positive + this many Negative
OUTDIR = Path("dataset_outputs")


# ================================================================================
# 2. IEDB / CEDAR EVIDENCE COLLECTION  (shared with PRIMARY + EXTERNAL)
# ================================================================================

def api_get(base_url: str, endpoint: str, params: Optional[dict] = None, timeout: int = 90) -> List[dict]:
    url = f"{base_url}/{endpoint}"
    response = requests.get(url, params=dict(params or {}), headers={"Accept": "application/json"}, timeout=timeout)
    response.raise_for_status()
    time.sleep(SLEEP_SECONDS)
    return response.json()


def fetch_all_pages(source_name: str, base_url: str, endpoint: str, params: Optional[dict] = None,
                     order: Optional[str] = None) -> pd.DataFrame:
    rows_all: List[dict] = []
    offset = 0
    while True:
        page_params = dict(params or {})
        page_params["limit"] = PAGE_SIZE
        page_params["offset"] = offset
        if order:
            page_params["order"] = order
        try:
            rows = api_get(base_url, endpoint, page_params)
        except requests.HTTPError:
            if "order" in page_params:
                page_params.pop("order", None)
                rows = api_get(base_url, endpoint, page_params)
            else:
                raise
        if not rows:
            break
        rows_all.extend(rows)
        print(f"    {source_name}/{endpoint}: fetched {len(rows_all)} rows")
        if len(rows) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
    return pd.DataFrame(rows_all)


def fetch_uniprot_sequence(uniprot_id: str) -> str:
    response = requests.get(f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json", timeout=60)
    response.raise_for_status()
    seq = response.json()["sequence"]["value"]
    print(f"[UniProt] {uniprot_id}: {len(seq)} amino acids")
    return seq


def normalize_sequence(value) -> Optional[str]:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except ValueError:
        pass
    seq = str(value).strip().upper()
    seq = re.sub(r"\s+", "", seq)
    if not re.fullmatch(r"[A-Z]+", seq):
        return None
    if any(aa not in STANDARD_AA for aa in seq):
        return None
    if not (MIN_LENGTH <= len(seq) <= MAX_LENGTH):
        return None
    return seq


def first_existing_column(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def flatten_values(value) -> List[str]:
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        raw = list(value)
    elif isinstance(value, str):
        text = value.strip()
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = json.loads(text)
                raw = parsed if isinstance(parsed, list) else [parsed]
            except json.JSONDecodeError:
                raw = [text]
        else:
            raw = [text]
    else:
        try:
            if pd.isna(value):
                return []
        except ValueError:
            pass
        raw = [value]
    return [str(x).strip().lower() for x in raw if str(x).strip()]


def outcome_from_values(values: Iterable[str]) -> str:
    values = list(values)
    if not values:
        return "Unknown"
    if any(v == "positive" for v in values):
        return "Positive"
    if any(v == "negative" for v in values):
        return "Negative"
    return "Other"


def endpoint_definitions() -> List[dict]:
    return [
        {"endpoint": "epitope_search", "order": "structure_id",
         "select": "structure_id,linear_sequence,linear_sequence_length,qualitative_measures,reference_ids,pubmed_ids",
         "sequence_columns": ["linear_sequence", "peptide_sequence", "sequence"],
         "outcome_columns": ["qualitative_measures", "qualitative_measure"]},
        {"endpoint": "tcell_search", "order": "tcell_id", "select": "*",
         "sequence_columns": ["linear_sequence", "peptide_sequence", "sequence", "structure_description", "epitope_linear_sequence"],
         "outcome_columns": ["qualitative_measure", "qualitative_measures"]},
        {"endpoint": "bcell_search", "order": "bcell_id", "select": "*",
         "sequence_columns": ["linear_sequence", "peptide_sequence", "sequence", "structure_description", "epitope_linear_sequence"],
         "outcome_columns": ["qualitative_measure", "qualitative_measures"]},
        {"endpoint": "mhc_search", "order": "elution_id", "select": "*",
         "sequence_columns": ["linear_sequence", "peptide_sequence", "sequence", "structure_description", "epitope_linear_sequence"],
         "outcome_columns": ["qualitative_measure", "qualitative_measures"]},
    ]


def query_attempts_for_antigen(uniprot_id: str, gene_symbol: str) -> List[dict]:
    """Filters used when we have a specific UniProt antigen (PRIMARY targets)."""
    return [
        {"label": "parent_source_antigen_iri_uniprot", "params": {"parent_source_antigen_iri": f"like.*{uniprot_id}*"}},
        {"label": "curated_source_antigen_iris_uniprot", "params": {"curated_source_antigen_iris": f"like.*{uniprot_id}*"}},
        {"label": "parent_source_antigen_names_gene", "params": {"parent_source_antigen_names": f"like.*{gene_symbol}*"}},
        {"label": "curated_source_antigen_names_gene", "params": {"curated_source_antigen_names": f"like.*{gene_symbol}*"}},
        {"label": "antigen_name_gene", "params": {"antigen_name": f"like.*{gene_symbol}*"}},
    ]


def query_attempts_for_organism(organism_query: str) -> List[dict]:
    """Filters used for a whole-organism pull (EXTERNAL targets, no single antigen)."""
    return [
        {"label": "parent_source_antigen_source_org_name", "params": {"parent_source_antigen_source_org_name": f"like.*{organism_query}*"}},
        {"label": "curated_source_antigen_source_org_names", "params": {"curated_source_antigen_source_org_names": f"like.*{organism_query}*"}},
        {"label": "host_organism_name", "params": {"parent_source_organism_name": f"like.*{organism_query}*"}},
    ]


def extract_evidence_rows(df: pd.DataFrame, database: str, endpoint: str, query_label: str,
                           sequence_columns: Iterable[str], outcome_columns: Iterable[str],
                           organism: str, protein_name: Optional[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    seq_col = first_existing_column(df, sequence_columns)
    if seq_col is None:
        return pd.DataFrame()
    outcome_col = first_existing_column(df, outcome_columns)
    assay_type = ENDPOINT_ASSAY_LABELS.get(endpoint, endpoint)
    rows = []
    for _, row in df.iterrows():
        peptide = normalize_sequence(row.get(seq_col))
        if peptide is None:
            continue
        raw_outcome = row.get(outcome_col) if outcome_col else None
        rows.append({
            "peptide": peptide,
            "seq_length": len(peptide),
            "assay_outcome": outcome_from_values(flatten_values(raw_outcome)),
            "database": database,
            "endpoint": endpoint,
            "assay_type": assay_type,
            "query_label": query_label,
            "organism": organism,
            "protein_name": protein_name,
        })
    out = pd.DataFrame(rows)
    print(f"    {database}/{endpoint}/{query_label}: extracted {len(out)} peptide evidence rows")
    return out


def collect_evidence_for_primary_target(source: dict, target: dict) -> pd.DataFrame:
    database, base_url = source["name"], source["base_url"]
    frames = []
    for ep in endpoint_definitions():
        for attempt in query_attempts_for_antigen(target["uniprot_id"], target["gene_symbol"]):
            params = dict(attempt["params"], select=ep["select"])
            try:
                df = fetch_all_pages(database, base_url, ep["endpoint"], params=params, order=ep["order"])
            except Exception as exc:
                print(f"    {database}/{ep['endpoint']}/{attempt['label']}: skipped ({exc})")
                continue
            frames.append(extract_evidence_rows(df, database, ep["endpoint"], attempt["label"],
                                                 ep["sequence_columns"], ep["outcome_columns"],
                                                 target["organism"], target["protein_name"]))
    frames = [f for f in frames if not f.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def collect_evidence_for_external_organism(source: dict, target: dict) -> pd.DataFrame:
    database, base_url = source["name"], source["base_url"]
    frames = []
    for ep in endpoint_definitions():
        for attempt in query_attempts_for_organism(target["organism_query"]):
            params = dict(attempt["params"], select=ep["select"])
            try:
                df = fetch_all_pages(database, base_url, ep["endpoint"], params=params, order=ep["order"])
            except Exception as exc:
                print(f"    {database}/{ep['endpoint']}/{attempt['label']}: skipped ({exc})")
                continue
            frames.append(extract_evidence_rows(df, database, ep["endpoint"], attempt["label"],
                                                 ep["sequence_columns"], ep["outcome_columns"],
                                                 target["organism"], None))
    frames = [f for f in frames if not f.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def verify_against_uniprot(evidence_df: pd.DataFrame, target: dict) -> pd.DataFrame:
    """Keep only peptides that are a verified substring of the antigen's reference sequence."""
    try:
        ref_seq = fetch_uniprot_sequence(target["uniprot_id"])
    except Exception as exc:
        print(f"[UniProt] {target['uniprot_id']}: fetch failed ({exc}); skipping verification for this antigen")
        return evidence_df
    mask = evidence_df["peptide"].apply(lambda p: p in ref_seq)
    kept = evidence_df[mask].copy()
    print(f"[verify] {target['protein_name']}: {mask.sum()}/{len(evidence_df)} peptides confirmed in UniProt sequence")
    return kept


def collapse_to_unique_pairs(evidence_df: pd.DataFrame) -> pd.DataFrame:
    """One row per (peptide, organism): Positive evidence wins over Negative when both exist."""
    if evidence_df.empty:
        return evidence_df

    def resolve(group: pd.DataFrame) -> pd.Series:
        outcomes = set(group["assay_outcome"])
        if "Positive" in outcomes and "Negative" in outcomes:
            label = "Conflicting"
        elif "Positive" in outcomes:
            label = "Positive"
        elif "Negative" in outcomes:
            label = "Negative"
        else:
            label = "Unknown"
        assays = "; ".join(sorted(set(group["assay_type"])))
        protein_name = group["protein_name"].dropna().iloc[0] if group["protein_name"].notna().any() else None
        return pd.Series({
            "seq_length": group["seq_length"].iloc[0],
            "resolved_outcome": label,
            "assay_type": assays,
            "protein_name": protein_name,
            "n_evidence_rows": len(group),
        })

    collapsed = (evidence_df.groupby(["peptide", "organism"], as_index=False)
                 .apply(resolve, include_groups=False)
                 .reset_index(drop=True))
    return collapsed


# ================================================================================
# 3. MHC BINDING ANNOTATION  (IEDB prediction tools API)
# ================================================================================

def _iedb_predict(url: str, method: str, sequence_text: str, allele: str, length: Optional[int] = None) -> pd.DataFrame:
    payload = {"method": method, "sequence_text": sequence_text, "allele": allele}
    if length is not None:
        payload["length"] = str(length)
    response = requests.post(url, data=payload, timeout=300)
    response.raise_for_status()
    text = response.text.strip()
    if not text or text.startswith("<"):
        return pd.DataFrame()
    from io import StringIO
    df = pd.read_csv(StringIO(text), sep="\t")
    if "error" in [c.lower() for c in df.columns]:
        return pd.DataFrame()
    return df


def annotate_mhc_binding(peptides: List[str], batch_size: int = 100) -> pd.DataFrame:
    unique_peptides = sorted(set(peptides))
    best_by_peptide: dict = {}

    by_length: dict = {}
    for p in unique_peptides:
        by_length.setdefault(len(p), []).append(p)

    # --- Class I ---
    for length, peps in sorted(by_length.items()):
        if not (8 <= length <= 14):
            continue
        for allele in MHC_I_ALLELES:
            for i in range(0, len(peps), batch_size):
                batch = peps[i:i + batch_size]
                fasta = "\n".join(f">p{j}\n{p}" for j, p in enumerate(batch))
                print(f"[MHC-I] {allele} len={length}: {len(batch)} peptides")
                try:
                    df = _iedb_predict(IEDB_MHCI_TOOL_URL, "netmhcpan_el", fasta, allele, length)
                except Exception as exc:
                    print(f"    failed: {exc}")
                    continue
                if df.empty:
                    continue
                seq_col = first_existing_column(df, ["peptide", "sequence"])
                rank_col = first_existing_column(df, ["percentile_rank", "percentile rank", "rank", "adjusted_rank"])
                ic50_col = first_existing_column(df, ["ic50", "affinity"])
                if seq_col is None or rank_col is None:
                    continue
                for _, row in df.iterrows():
                    pep = row[seq_col]
                    rank = row[rank_col]
                    if pd.isna(rank):
                        continue
                    prev = best_by_peptide.get(pep)
                    if prev is None or rank < prev["MHC_Percentile_Rank"]:
                        best_by_peptide[pep] = {
                            "MHC_Class_Used": "I",
                            "MHC_Percentile_Rank": float(rank),
                            "MHC_IC50_nM": float(row[ic50_col]) if ic50_col and pd.notna(row.get(ic50_col)) else np.nan,
                            "MHC_Allele_Best": allele,
                        }

    # --- Class II ---
    for length, peps in sorted(by_length.items()):
        if not (11 <= length <= 30):
            continue
        for allele in MHC_II_ALLELES:
            for i in range(0, len(peps), batch_size):
                batch = peps[i:i + batch_size]
                fasta = "\n".join(f">p{j}\n{p}" for j, p in enumerate(batch))
                print(f"[MHC-II] {allele} len={length}: {len(batch)} peptides")
                try:
                    df = _iedb_predict(IEDB_MHCII_TOOL_URL, "netmhciipan_el", fasta, allele, length)
                except Exception as exc:
                    print(f"    failed: {exc}")
                    continue
                if df.empty:
                    continue
                seq_col = first_existing_column(df, ["peptide", "sequence"])
                rank_col = first_existing_column(df, ["percentile_rank", "percentile rank", "rank", "adjusted_rank"])
                ic50_col = first_existing_column(df, ["ic50", "affinity"])
                if seq_col is None or rank_col is None:
                    continue
                for _, row in df.iterrows():
                    pep = row[seq_col]
                    rank = row[rank_col]
                    if pd.isna(rank):
                        continue
                    prev = best_by_peptide.get(pep)
                    if prev is None or rank < prev["MHC_Percentile_Rank"]:
                        best_by_peptide[pep] = {
                            "MHC_Class_Used": "II",
                            "MHC_Percentile_Rank": float(rank),
                            "MHC_IC50_nM": float(row[ic50_col]) if ic50_col and pd.notna(row.get(ic50_col)) else np.nan,
                            "MHC_Allele_Best": allele,
                        }

    results = []
    n_annotated = 0
    for pep in unique_peptides:
        rec = best_by_peptide.get(pep, {
            "MHC_Class_Used": np.nan, "MHC_Percentile_Rank": np.nan,
            "MHC_IC50_nM": np.nan, "MHC_Allele_Best": np.nan,
        })
        if pep in best_by_peptide:
            n_annotated += 1
        rec["peptide"] = pep
        results.append(rec)
    print(f"[MHC prediction] annotated {n_annotated}/{len(unique_peptides)} peptides")

    # --- Fill in IC50 (nM) for the winning (peptide, allele) pairs -------------

    ba_groups: dict = {}   # (mhc_class, allele, length) -> [peptides]
    for pep, rec in best_by_peptide.items():
        key = (rec["MHC_Class_Used"], rec["MHC_Allele_Best"], len(pep))
        ba_groups.setdefault(key, []).append(pep)

    for (mhc_class, allele, length), peps in ba_groups.items():
        url = IEDB_MHCI_TOOL_URL if mhc_class == "I" else IEDB_MHCII_TOOL_URL
        method = "netmhcpan_ba" if mhc_class == "I" else "netmhciipan_ba"
        for i in range(0, len(peps), batch_size):
            batch = peps[i:i + batch_size]
            fasta = "\n".join(f">p{j}\n{p}" for j, p in enumerate(batch))
            print(f"[MHC-{mhc_class} IC50] {allele} len={length}: {len(batch)} peptides")
            try:
                df = _iedb_predict(url, method, fasta, allele, length)
            except Exception as exc:
                print(f"    failed: {exc}")
                continue
            if df.empty:
                continue
            seq_col = first_existing_column(df, ["peptide", "sequence"])
            ic50_col = first_existing_column(df, ["ic50", "affinity"])
            if seq_col is None or ic50_col is None:
                continue
            ic50_by_pep = dict(zip(df[seq_col], df[ic50_col]))
            for rec in results:
                if rec["peptide"] in ic50_by_pep and pd.notna(ic50_by_pep[rec["peptide"]]):
                    rec["MHC_IC50_nM"] = float(ic50_by_pep[rec["peptide"]])

    return pd.DataFrame(results)


# ================================================================================
# 4. PIPELINE
# ================================================================================

def run_collection(outdir: Path) -> pd.DataFrame:
    outdir.mkdir(parents=True, exist_ok=True)
    all_evidence = []

    print("=" * 72)
    print("PRIMARY collection (12 named antigens, UniProt-verified)")
    print("=" * 72)
    for target in PRIMARY_TARGETS:
        for source in SOURCES:
            print(f"\n--- {source['name']} :: {target['protein_name']} ---")
            ev = collect_evidence_for_primary_target(source, target)
            if not ev.empty:
                ev = verify_against_uniprot(ev, target)
                ev["target_group"] = "PRIMARY"
                all_evidence.append(ev)

    print("\n" + "=" * 72)
    print("EXTERNAL collection (organism-level: Dengue, Zika, HCV, RSV, HKU15)")
    print("=" * 72)
    for target in EXTERNAL_ORGANISMS:
        for source in SOURCES:
            print(f"\n--- {source['name']} :: {target['organism']} ---")
            ev = collect_evidence_for_external_organism(source, target)
            if not ev.empty:
                ev["target_group"] = "EXTERNAL"
                all_evidence.append(ev)

    evidence_df = pd.concat([f for f in all_evidence if not f.empty], ignore_index=True)
    evidence_df.to_csv(outdir / "raw_evidence.csv", index=False)
    print(f"\n[saved] raw evidence -> {outdir / 'raw_evidence.csv'} ({len(evidence_df)} rows)")

    collapsed = collapse_to_unique_pairs(evidence_df)
    group_lookup = evidence_df.drop_duplicates(["peptide", "organism"])[["peptide", "organism", "target_group"]]
    collapsed = collapsed.merge(group_lookup, on=["peptide", "organism"], how="left")
    collapsed = collapsed[collapsed["resolved_outcome"].isin(["Positive", "Negative"])].reset_index(drop=True)
    collapsed.to_csv(outdir / "unique_labeled_peptides.csv", index=False)
    print(f"[saved] unique labeled peptides -> {outdir / 'unique_labeled_peptides.csv'} ({len(collapsed)} rows)")
    return collapsed


def assign_pool_ids(collapsed: pd.DataFrame, seed: int) -> pd.DataFrame:

    rng = random.Random(seed)
    collapsed = collapsed.copy()
    order = list(collapsed.index)
    rng.shuffle(order)
    counters = {"PRIMARY": 0, "EXTERNAL": 0}
    id_map = {}
    for idx in order:
        grp = collapsed.loc[idx, "target_group"]
        counters[grp] += 1
        id_map[idx] = f"{grp}_{counters[grp]:05d}"
    collapsed["dataset_id"] = collapsed.index.map(id_map)
    return collapsed


def build_balanced_dataset(collapsed: pd.DataFrame, target_per_class: int, seed: int,
                            exclude_dataset_ids: Optional[set] = None) -> pd.DataFrame:
    if exclude_dataset_ids:
        before = len(collapsed)
        collapsed = collapsed[~collapsed["dataset_id"].isin(exclude_dataset_ids)]
        print(f"[exclude] removed {before - len(collapsed)} peptides already used in a prior split "
              f"(e.g. keep this run non-overlapping with an earlier train/main export)")
    rng = random.Random(seed)
    pos = collapsed[collapsed["resolved_outcome"] == "Positive"]
    # (dataset_id was already assigned on the full pool by assign_pool_ids)
    neg = collapsed[collapsed["resolved_outcome"] == "Negative"]

    def stratified_sample(df: pd.DataFrame, n: int) -> pd.DataFrame:
        if len(df) <= n:
            return df
        # organism-stratified so no single organism dominates the draw
        frac = n / len(df)
        parts = []
        for _, grp in df.groupby("organism"):
            take = max(1, round(len(grp) * frac))
            idx = list(grp.index)
            rng.shuffle(idx)
            parts.append(grp.loc[idx[:take]])
        out = pd.concat(parts)
        if len(out) > n:
            idx = list(out.index)
            rng.shuffle(idx)
            out = out.loc[idx[:n]]
        return out

    pos_sampled = stratified_sample(pos, target_per_class)
    neg_sampled = stratified_sample(neg, target_per_class)

    pos_sampled = pos_sampled.assign(label=1)
    neg_sampled = neg_sampled.assign(label=0)
    combined = pd.concat([pos_sampled, neg_sampled], ignore_index=True)
    print(f"[balance] kept {len(pos_sampled)} Positive + {len(neg_sampled)} Negative = {len(combined)} rows")
    return combined


def finalize_dataset(df: pd.DataFrame, seed: int) -> pd.DataFrame:

    mhc = annotate_mhc_binding(df["peptide"].tolist())
    df = df.merge(mhc, on="peptide", how="left")

    df["Class_I"] = (df["MHC_Class_Used"] == "I").astype(int)
    df["Class_II"] = (df["MHC_Class_Used"] == "II").astype(int)

    # dataset_id is already assigned (stable, from the full pool -- see assign_pool_ids)
    out = pd.DataFrame({
        "dataset_id": df["dataset_id"],
        "Peptide": df["peptide"],
        "Organism": df["organism"],
        "Assay": df["assay_type"],
        "label": df["label"],
        "seq_length": df["seq_length"],
        "MHC_Class_Used": df["MHC_Class_Used"],
        "MHC_Percentile_Rank": df["MHC_Percentile_Rank"],
        "MHC_IC50_nM": df["MHC_IC50_nM"],
        "MHC_Allele_Best": df["MHC_Allele_Best"],
        "Class_I": df["Class_I"],
        "Class_II": df["Class_II"],
    })
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)


def run_full_pipeline(outdir: Path, target_per_class: int, seed: int,
                       output_name: str = "Data_collection_2k.csv",
                       exclude_from: Optional[str] = None,
                       pool_cache: Optional[str] = None) -> pd.DataFrame:

    random.seed(seed)
    np.random.seed(seed)
    outdir.mkdir(parents=True, exist_ok=True)

    if pool_cache:
        collapsed = pd.read_csv(pool_cache)
        print(f"[pool] reusing cached numbered pool -> {pool_cache} ({len(collapsed)} peptides)")
    else:
        collapsed = run_collection(outdir)
        collapsed = assign_pool_ids(collapsed, seed)
        collapsed.to_csv(outdir / "numbered_pool.csv", index=False)
        print(f"[saved] full numbered pool -> {outdir / 'numbered_pool.csv'} "
              f"(re-use this with --pool-cache to draw another split, e.g. a validation set)")

    exclude_ids = None
    if exclude_from:
        prior = pd.read_csv(exclude_from)
        exclude_ids = set(prior["dataset_id"])
        print(f"[exclude] loaded {len(exclude_ids)} dataset_id values to keep out of this split -> {exclude_from}")

    balanced = build_balanced_dataset(collapsed, target_per_class, seed, exclude_dataset_ids=exclude_ids)
    final_df = finalize_dataset(balanced, seed)

    out_path = outdir / output_name
    final_df.to_csv(out_path, index=False)
    print(f"\n[saved] final dataset -> {out_path} ({len(final_df)} rows)")
    print(final_df["label"].value_counts())
    return final_df


# ================================================================================
# 5. CLI
# ================================================================================

def _parse_cli_args():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--outdir", default=str(OUTDIR), help=f"Output directory (default: {OUTDIR})")
    ap.add_argument("--target-per-class", type=int, default=TARGET_TOTAL_PER_CLASS,
                     help=f"Final Positive count and Negative count each (default: {TARGET_TOTAL_PER_CLASS})")
    ap.add_argument("--seed", type=int, default=RANDOM_SEED, help=f"Random seed (default: {RANDOM_SEED})")
    ap.add_argument("--output-name", default="Data_collection_reconstructed.csv",
                     help="Filename for the final exported CSV (default: %(default)s)")
    ap.add_argument("--pool-cache", default=None,
                     help="Path to a previously saved numbered_pool.csv; skips re-querying IEDB/CEDAR "
                          "and draws a new split from that same numbered pool.")
    ap.add_argument("--exclude-from", default=None,
                     help="Path to a previously exported final CSV; excludes those dataset_id values "
                          "from this draw, so e.g. a validation split has no peptide overlap with the main split.")
    args, _unknown = ap.parse_known_args()
    return args


if __name__ == "__main__":
    _args = _parse_cli_args()
    run_full_pipeline(Path(_args.outdir), _args.target_per_class, _args.seed,
                       output_name=_args.output_name, exclude_from=_args.exclude_from,
                       pool_cache=_args.pool_cache)